# Slit flow benchmark

This notebook builds a simple 3D slit, runs the bundled single-phase solver, recovers the velocity profile, and checks that the measured permeability follows the cubic law.

In [ ]:
from pathlib import Path
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

cwd = Path.cwd().resolve()
if (cwd / "benchmarks").exists():
    repo_root = cwd
    benchmark_dir = cwd / "benchmarks"
elif cwd.name == "benchmarks":
    benchmark_dir = cwd
    repo_root = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the repo root or from benchmarks/.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(benchmark_dir)
from latteasy.preprocessing.IO_tools import LattEasySimulation

benchmark_dir

In [ ]:
length = 96
span = 160
height = 64
aperture = 12
buffer_layers = 0
periodic = (False, True, False)
num_cores = 4

print(f"length={length}, span={span}, height={height}, aperture={aperture}, periodic={periodic}, cores={num_cores}")

## Geometry

The array is stored as `(span, length, height)` so the current single-phase wrapper sends the slit along the flow direction. The slit is periodic in the spanwise direction, and no extra open buffer slices are added.

In [ ]:
geometry = np.zeros((span, length, height), dtype=bool)
gap_start = (height - aperture) // 2
gap_stop = gap_start + aperture
geometry[:, :, gap_start:gap_stop] = True

cmap = ListedColormap(["#4a3323", "#d7eefb"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True, dpi=180)

axes[0].imshow(geometry[span // 2].T, origin="lower", cmap=cmap, interpolation="nearest")
axes[0].set_title("Mid-span slice")
axes[0].set_xlabel("Flow direction")
axes[0].set_ylabel("Height")

axes[1].imshow(geometry[:, length // 2, :].T, origin="lower", cmap=cmap, interpolation="nearest")
axes[1].set_title("Cross-section")
axes[1].set_xlabel("Span")
axes[1].set_ylabel("Height")

plt.show()
plt.close(fig)

## Run the simulation

In [ ]:
simulation = LattEasySimulation(
    geometry,
    buffer_layers=buffer_layers,
    cpus=num_cores,
    periodic=periodic,
)
permeability = simulation.run_sim()

folder = Path(simulation.folder_path).resolve()
output_dir = folder / "output"

print(f"Permeability: {permeability:.5g} [l.u.^2]")
print(f"Simulation folder: {folder}")

## Recover the profile and compare with the cubic law

The solver reports Darcy permeability using the full domain cross-section, so the slit benchmark is

`k = b^3 / (12 H)`

for slit aperture `b` inside a box of total height `H`.

In [ ]:
vel_path = sorted(output_dir.glob("*_vel.dat"))[-1]
shape_match = re.search(r"_(\d+)_(\d+)_(\d+)_vel\.dat$", vel_path.name)
nx, ny, nz = [int(value) for value in shape_match.groups()]

# The raw velocity field is written in row-major order.
flat_velocity = np.loadtxt(vel_path).reshape(-1, 3)
velocity = np.stack([
    flat_velocity[:, 0].reshape((nx, ny, nz), order="C"),
    flat_velocity[:, 1].reshape((nx, ny, nz), order="C"),
    flat_velocity[:, 2].reshape((nx, ny, nz), order="C"),
], axis=-1)

delta_p = float(re.search(r"<press>\s*([0-9.eE+-]+)\s*</press>", (folder / "pore.xml").read_text()).group(1))
nu = 1 / 6
pressure_gradient = delta_p / (nx - 1)

ux = velocity[..., 0]
mid_span = ny // 2
core_x = slice(None) if buffer_layers == 0 else slice(buffer_layers, -buffer_layers)
slice_ux = ux[core_x, mid_span, :].T
profile = ux[core_x, :, gap_start:gap_stop].mean(axis=(0, 1))

z = np.arange(aperture) + 0.5
profile_theory = pressure_gradient / (2 * nu) * z * (aperture - z)
k_theory = aperture**3 / (12 * height)
k_error = (permeability - k_theory) / k_theory

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True, dpi=180)

image = axes[0].imshow(slice_ux, origin="lower", cmap="magma")
axes[0].axhline(gap_start - 0.5, color="white", linestyle="--", linewidth=1)
axes[0].axhline(gap_stop - 0.5, color="white", linestyle="--", linewidth=1)
axes[0].set_title("Recovered flow field")
axes[0].set_xlabel("Flow direction")
axes[0].set_ylabel("Height")
fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.03, label="u_x [l.u.]")

axes[1].plot(profile, z, "o-", color="#0b6e99", label="LBM")
axes[1].plot(profile_theory, z, color="#d1495b", linewidth=2, label="cubic law")
axes[1].set_title("Parabolic slit profile")
axes[1].set_xlabel("u_x [l.u.]")
axes[1].set_ylabel("Cell-center height in slit")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

plt.show()
plt.close(fig)

print(f"Measured permeability: {permeability:.5g} [l.u.^2]")
print(f"Cubic-law permeability: {k_theory:.5g} [l.u.^2]")
print(f"Relative error: {k_error:+.2%}")